# Welcome to the Python Yard (beta)

This page provides a python notebook that runs entirely in your browser with no account and no server. Sort of like a simplified version of Google Colab or Jupyter Notebook.

> **Beta:** the Python Yard is new and still being tested, so expect rough edges. If something breaks, download your work first (**File → Download**), then let your instructor know.

- **Run a cell:** click it and press <kbd>Shift</kbd>+<kbd>Enter</kbd>. The first run might take a while because your browser will need to download Python.
- **Your work is not saved automatically.** It will disappear from clearing site data, using a private window, or switching to a different computer. To keep a copy, use **File → Download**.
- **Course data:** any CSV in the [course datasets folder](https://github.com/jackbandy/data-science-fun/tree/main/datasets) can be read with `pl.read_csv(url)`, using the file's **Raw** link.
- **Lecture notebooks**: File → Open… lists every `weekN.ipynb`.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# Lets pl.read_csv() download from a URL inside the browser.
import pyodide_http
pyodide_http.patch_all()

In [ ]:
print("Hello, Yard!")

## Demo: a year at UIC-Halsted

Click the arrow next to this heading to open the demo. It reads a course dataset (daily entries at the UIC-Halsted Blue Line station in 2025) plots it, then splits it into a four-part decomposition like the week 6 slides.

In [ ]:
# Daily entries at UIC-Halsted (Blue Line), every day of 2025
url = "https://raw.githubusercontent.com/jackbandy/data-science-fun/main/datasets/cta-ridership/Station_Entries_UIC-Halsted_2025.csv"
uic = (
    pl.read_csv(url, schema_overrides={"rides": pl.String})
    .with_columns(
        pl.col("date").str.to_date("%m/%d/%Y"),
        pl.col("rides").str.replace_all(",", "").cast(pl.Int64),
    )
    .sort("date")
)
uic.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(uic["date"], uic["rides"], color="#0086B8", linewidth=1.5)  # CTA Blue Line blue
ax.set_title("Daily entries at UIC-Halsted (Blue Line), 2025", loc="left")
ax.set_ylabel("Entries per day")
ax.set_ylim(bottom=0)
ax.grid(axis="y", color="#e5e5e5")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### Four-part decomposition

An additive model splits the series into parts that add back up to it: **rides = trend + seasonal + noise**.

- **Level** and **trend**: the baseline and its slow drift (`seasonal_decompose` returns them together as `trend`)
- **Seasonality**: the weekly pattern that repeats every 7 days
- **Noise**: whatever is left over

Run the cells below one at a time.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

rides = uic["rides"].to_numpy().astype(float)
result = seasonal_decompose(rides, model="additive", period=7)

In [ ]:
# The first and last 3 days are blank: a centered 7-day average needs 3 days on each side
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(uic["date"], result.trend, color="#0086B8", linewidth=1.5)
ax.set_title("Trend: a 7-day moving average (the level is part of it)", loc="left")
ax.set_ylabel("Entries per day")
ax.set_ylim(bottom=0)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(uic["date"][:28], result.seasonal[:28], marker="o", color="#0086B8")
ax.axhline(0, color="gray", linestyle="--")
ax.set_title("Seasonality: the same weekly wave, repeated all year (first four weeks shown)", loc="left")
ax.set_ylabel("Entries vs. trend")
plt.show()

In [ ]:
# The deepest dips are holidays the weekly pattern can't know about (MLK Day, Labor Day, Thanksgiving)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(uic["date"], result.resid, color="#0086B8", linewidth=1)
ax.axhline(0, color="gray", linestyle="--")
ax.set_title("Noise: what trend and seasonality leave unexplained", loc="left")
ax.set_ylabel("Entries vs. trend + seasonal")
plt.show()

In [ ]:
# All four stacked, like the slides: observed = trend + seasonal + noise
fig = result.plot()
fig.set_size_inches(10, 6)
fig.get_axes()[-1].set_xlabel("Day of 2025")
plt.show()

## Tinker zone

Add a cell with the **+** button (or press <kbd>Esc</kbd> then <kbd>B</kbd>), and run it with <kbd>Shift</kbd>+<kbd>Enter</kbd>.